Get API KEY from [Cerebras Website]("https://www.cerebras.ai/"). Save to environment as "CEREBRAS_API_KEY".

See README for more details

In [1]:
import os
from cerebras.cloud.sdk import Cerebras

client = Cerebras(
    api_key=os.environ.get("CEREBRAS_API_KEY")
)

In [2]:
SYSTEM_PROMPT_STRING = """
Your task is to partition the 16 words into 4 groups of 4 words/phrases based on shared connections. 
Output requirements (STRICT): 
OUTPUT ONLY the final groups of words/phrases.
Do NOT provide reasoning or explanations under any circumstances.
DO NOT output any text other than the 4 groups.
Use ONLY the EXACT words/phrases from the puzzle.
Make sure there are EXACTLY 4 groups of 4 words/phrases each with their category names. NO EXCEPTIONS.
Return the answer exactly in this format:

GROUP 1: word1 || word2 || word3 || word4
GROUP 2: word1 || word2 || word3 || word4
GROUP 3: word1 || word2 || word3 || word4
GROUP 4: word1 || word2 || word3 || word4
"""

In [3]:
import re

def call_llm(system_prompt, user_prompt,
                        model = "llama3.1-8b",
                        temperature = 0.1,
                        max_tokens = 600):
    """
    Sends a chat request to the Cerebras API and returns the response content.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=temperature,
            max_tokens=max_tokens
        )
        msg = response.choices[0].message
        return (msg.content or "").strip()
    
    except Exception as e:
        return f"ERROR: {e}"

def convert_puzzle_to_prompt(words16):
    word_list_str = " || ".join(words16)
    return f"Here are the 16 words: {word_list_str}"

def parse_response_to_pred_groups(response):
    pattern = r"GROUP \d+: (.+)"
    
    groups = re.findall(pattern, response)
    parsed_groups = [ [word.strip() for word in group.split("||")] for group in groups ]
    
    return parsed_groups

In [4]:
from data_loader import get_train_test_split

ds_train, ds_test = get_train_test_split()
print("Training puzzles:", len(ds_train))
print("Testing puzzles:", len(ds_test))

c:\Users\kunri\miniconda3\envs\cs175\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Training puzzles: 521
Testing puzzles: 131


In [5]:
import random

#Number of times LLM can retry if it hallucinates before marking it as an error
MAX_RETRIES = 4

def valid_prediction(pred_groups, words16):
    if pred_groups is None:
        return False
    if len(pred_groups) != 4:
        return False
    if any(len(group) != 4 for group in pred_groups):
        return False
    
    pred_words = [word for group in pred_groups for word in group]
    if set(pred_words) == set(words16):
        return True
    else:
        return False

def make_few_shot_prompt(words16, k, split_for_few_shot):
    few_shot_prompt = "Here are some previous examples:\n"
    count = 0
    for fs_row in split_for_few_shot:
        fs_words = fs_row["words"]
        # Skip because of leakage
        if set(fs_words) == set(words16):
            continue
        fs_groups = [ans["words"] for ans in fs_row["answers"]]
        fs_text = f"Here are 16 words: {' || '.join(fs_words)}\n"
        for i, group in enumerate(fs_groups, 1):
            fs_text += f"GROUP {i}: {' || '.join(group)}\n"
        few_shot_prompt += fs_text + "\n"
        count += 1
        if count >= k:
            break
    return few_shot_prompt

def solve_puzzle(words16, k=0, split_for_few_shot=None, model="llama3.1-8b", temperature=0.1, max_tokens=600, max_retries=MAX_RETRIES):
    few_shot_prompt = ""
    if k > 0 and split_for_few_shot is not None:
        few_shot_examples = random.sample(list(split_for_few_shot), k)
        few_shot_prompt = make_few_shot_prompt(words16, k, few_shot_examples)
    user_prompt = few_shot_prompt + "NOW, solve this puzzle and only this one: " + convert_puzzle_to_prompt(words16)

    attempt = 0
    while attempt <= max_retries:
        response = call_llm(
            SYSTEM_PROMPT_STRING,
            user_prompt,
            model=model,
            temperature=(temperature + 0.1*attempt),  
            max_tokens=max_tokens
        )

        pred_groups = parse_response_to_pred_groups(response)

        if valid_prediction(pred_groups, words16):
            return pred_groups

        print(f"Invalid LLM output: {response}\nCorrect Answer: {words16}\nRetrying ({attempt+1}/{max_retries})")
        attempt += 1

    print("ERROR: LLM failed after retries")
    return []

In [6]:
row0 = ds_train[0]
words16 = row0["words"]
pred_groups = solve_puzzle(words16)

print("\nPredicted groups:")
for g in pred_groups:
    print(g)

print("\nGold groups:")
for ans in row0["answers"]:
    print(ans["answerDescription"], "->", ans["words"])


Predicted groups:
['GIANT', 'JUMBO', 'MONSTER', 'SUPER']
['BOWL', 'LADLE', 'POT', 'SPOON']
['CHARACTER', 'INDIVIDUAL', 'PARTY', 'PERSON']
['ALIEN', 'AVATAR', 'DUNE', 'TRON']

Gold groups:
MASSIVE -> ['GIANT', 'JUMBO', 'MONSTER', 'SUPER']
USED WHEN SERVING SOUP -> ['BOWL', 'LADLE', 'POT', 'SPOON']
SOMEBODY -> ['CHARACTER', 'INDIVIDUAL', 'PARTY', 'PERSON']
SCI-FI FRANCHISES -> ['ALIEN', 'AVATAR', 'DUNE', 'TRON']


The next 3 code blocks are deprecated. I only kept them because they were used to generate   "LLM_few_shot_results.csv". The code blocks after are used to create the JSON result files

In [ ]:
from conn import accuracy_min_swaps, accuracy_zero_one, evaluate
from data_loader import gold_groups_from_row
import time

N_EVAL = len(ds_test)

results_list = []

few_shot_values = [0, 1, 3, 5]
#few_shot_values = [10, 20]
for k in few_shot_values:

    start = time.time()

    results = evaluate(
        ds_test,
        metric_fns={
            "zero_one": accuracy_zero_one,
            "min_swaps": accuracy_min_swaps,
        },
        solver_fn=lambda words: solve_puzzle(words, k=k, split_for_few_shot=ds_train),
        max_samples=N_EVAL,
        gold_from_row=gold_groups_from_row,
        verbose=True,
    )
    acc, n_eval = results["zero_one"]
    mean_swaps, _ = results["min_swaps"]

    end = time.time()
    runtime = end - start

    results_list.append({
        "k": k,
        "zero_one_accuracy": acc,
        "mean_swaps": mean_swaps,
        "runtime": runtime,
        "n_eval": n_eval
    })

    print(f"k={k} | Zero-one accuracy: {acc:.4f}  (n={n_eval}, requested={N_EVAL})")
    print(f"k={k} | Mean 1-1 swaps to correct: {mean_swaps:.2f}  (n={n_eval})")
    print(f"k={k} | Runtime: {runtime:.2f} seconds")


In [ ]:
N_EVAL = len(ds_test)

results_list = []

#few_shot_values = [0, 1, 3, 5]
few_shot_values = [10, 20]
for k in few_shot_values:

    start = time.time()

    results = evaluate(
        ds_test,
        metric_fns={
            "zero_one": accuracy_zero_one,
            "min_swaps": accuracy_min_swaps,
        },
        solver_fn=lambda words: solve_puzzle(words, k=k, split_for_few_shot=ds_train),
        max_samples=N_EVAL,
        gold_from_row=gold_groups_from_row,
        verbose=True,
    )
    acc, n_eval = results["zero_one"]
    mean_swaps, _ = results["min_swaps"]

    end = time.time()
    runtime = end - start

    results_list.append({
        "k": k,
        "zero_one_accuracy": acc,
        "mean_swaps": mean_swaps,
        "runtime": runtime,
        "n_eval": n_eval
    })

    print(f"k={k} | Zero-one accuracy: {acc:.4f}  (n={n_eval}, requested={N_EVAL})")
    print(f"k={k} | Mean 1-1 swaps to correct: {mean_swaps:.2f}  (n={n_eval})")
    print(f"k={k} | Runtime: {runtime:.2f} seconds")


In [ ]:
import pandas as pd

df_results = pd.DataFrame(results_list)
df_results.to_csv("LLM_few_shot_results.csv", index=False)
display(df_results)

Above evaluation code is deprecated

In [12]:
from conn import (
    accuracy_min_swaps,
    accuracy_zero_one,
    correct_word_count,
    evaluate,
    n_correct_groups,
)
from data_loader import gold_groups_from_row

def _build_output_row(i, row, words16, gold, pred, z, s, inference_time_sec=0.0):
    valid = len(pred) == 4 and all(len(g) == 4 for g in pred)
    if not valid:
        return None
    return {
        "index": i,
        "words16": words16,
        "pred_groups": pred,
        "gold_groups": gold,
        "zero_one": z,
        "min_swaps": s,
        "date": str(row.get("date", "")),
        "levels": row.get("levels", []),
        "n_correct_groups": n_correct_groups(pred, gold),
        "correct_word_count": correct_word_count(pred, gold),
        "inference_time_sec": inference_time_sec,
        "is_valid": True,
    }

This code block should be run with few_shot_values = [0, 1, 3, 5, 10] first. Running with k=20 can reach the token limit for Cerebras' API

In [ ]:
import json
import time
from pathlib import Path

SAVE_RESULTS = True
N_EVAL = len(ds_test)
few_shot_values = [0, 1, 3, 5, 10]
#few_shot_values = [20]

if SAVE_RESULTS:
    run_dir = globals().get("run_dir") or Path("reports/LLM")
    run_dir = Path(run_dir)
    run_dir.mkdir(parents=True, exist_ok=True)

    for k in few_shot_values:
        outputs_test = []
        n_test = min(N_EVAL, len(ds_test))

        for i in range(n_test):
            row = ds_test[i]
            words16 = row.get("words", [])
            if len(words16) != 16:
                continue
            gold = gold_groups_from_row(row)
            if len(gold) != 4:
                continue

            t0 = time.perf_counter()
            pred = solve_puzzle(words16, k=k, split_for_few_shot=ds_train)
            inference_sec = time.perf_counter() - t0

            z = accuracy_zero_one(pred, gold)
            s = accuracy_min_swaps(pred, gold)
            rec = _build_output_row(i, row, words16, gold, pred, z, s, inference_sec)
            if rec is not None:
                outputs_test.append(rec)

            if (i + 1) % 10 == 0:
                print(f"[k={k}] Processed {i + 1}/{n_test}")

        if outputs_test:
            acc_t = sum(o["zero_one"] for o in outputs_test) / len(outputs_test)
            mean_swaps_t = sum(o["min_swaps"] for o in outputs_test) / len(outputs_test)
            n_cg_t = sum(o["n_correct_groups"] for o in outputs_test) / len(outputs_test)
            cwc_t = sum(o["correct_word_count"] for o in outputs_test) / len(outputs_test)
            mean_inference_sec_t = sum(o["inference_time_sec"] for o in outputs_test) / len(outputs_test)
            total_inference_sec_t = sum(o["inference_time_sec"] for o in outputs_test)

            metrics_test = {
                "k": k,
                "model": "llama3.1-8b",
                "split": "test",
                "zero_one_accuracy": acc_t,
                "mean_min_swaps": mean_swaps_t,
                "mean_n_correct_groups": n_cg_t,
                "mean_correct_word_count": cwc_t,
                "mean_inference_time_sec": mean_inference_sec_t,
                "total_inference_time_sec": total_inference_sec_t,
                "n_eval": len(outputs_test),
            }

            (run_dir / f"llm_k{k}_test_metrics.json").write_text(json.dumps(metrics_test, indent=2))
            (run_dir / f"llm_k{k}_test_outputs.json").write_text(json.dumps(outputs_test, indent=2))
            print(f"Saved k={k} test results to {run_dir}/ (llm_k{k}_test_*)")
            print(f"k={k} | Test zero-one: {acc_t:.4f}  mean min_swaps: {mean_swaps_t:.2f}  n={len(outputs_test)}")
        else:
            print(f"k={k} | No valid test outputs to save.")

else:
    for k in few_shot_values:
        results_test = evaluate(
            ds_test,
            metric_fns={"zero_one": accuracy_zero_one, "min_swaps": accuracy_min_swaps},
            solver_fn=lambda words: solve_puzzle(words, k=k, split_for_few_shot=ds_train),
            max_samples=N_EVAL,
            gold_from_row=gold_groups_from_row,
        )
        acc_t, n_t = results_test["zero_one"]
        mean_swaps_t, _ = results_test["min_swaps"]
        print(f"k={k} | Test zero-one: {acc_t:.4f}  (n={n_t})")
        print(f"k={k} | Test mean min_swaps: {mean_swaps_t:.2f}")

[k=10] Processed 10/131
[k=10] Processed 20/131
[k=10] Processed 30/131
[k=10] Processed 40/131
[k=10] Processed 50/131
[k=10] Processed 60/131
[k=10] Processed 70/131
[k=10] Processed 80/131
[k=10] Processed 90/131
[k=10] Processed 100/131
[k=10] Processed 110/131
[k=10] Processed 120/131
[k=10] Processed 130/131
Saved k=10 test results to results\2026-03-16_01-29-43/ (llm_k10_test_*)
k=10 | Test zero-one: 0.9618  mean min_swaps: 0.05  n=131
[k=20] Processed 10/131
[k=20] Processed 20/131
[k=20] Processed 30/131
[k=20] Processed 40/131
[k=20] Processed 50/131
[k=20] Processed 60/131
[k=20] Processed 70/131
[k=20] Processed 80/131
[k=20] Processed 90/131
[k=20] Processed 100/131
[k=20] Processed 110/131
[k=20] Processed 120/131
[k=20] Processed 130/131
Saved k=20 test results to results\2026-03-16_01-29-43/ (llm_k20_test_*)
k=20 | Test zero-one: 0.9695  mean min_swaps: 0.05  n=131
